# 🎧 Higgs Audio v3 — Colab benchmark (CUDA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/higgs_colab_benchmark.ipynb)

Сравнение серверного GPU с локальным замером на Apple Silicon M1. Блокнот следует
правилам `AGENTS.md`: **ни один результат не печатается, если операция не выполнялась.**
Незапущенный или недоступный этап отмечается `SKIPPED` с указанием причины.

## Как устроена изоляция памяти

Каждая модель запускается **отдельным процессом в отдельном окружении**. Ядро блокнота
никогда не держит ссылку на модель, поэтому VRAM и хостовая RAM освобождаются
завершением процесса, а не вызовом `del`.

| Этап | Окружение | Стек |
|---|---|---|
| STT `bosonai/higgs-audio-v3-stt` | venv `--system-site-packages` | `transformers==4.51.0` + remote code чекпоинта |
| TTS `bosonai/higgs-tts-3-4b` | отдельный venv | SGLang-Omni (`sgl-omni serve`) |

Два окружения обязательны: STT требует `transformers` 4.51 и свой remote code, а TTS —
`transformers` 5.x и стек SGLang-Omni. В одном окружении они несовместимы.

## Что реально доступно для TTS

В репозитории `bosonai/higgs-tts-3-4b` **нет `.py`-файлов**, а архитектура
`higgs_multimodal_qwen3` **не реализована в `transformers`**, поэтому пути
«`from_pretrained` + `generate`» не существует. Единственная документированная
first-party реализация под CUDA — `sglang_omni/models/higgs_tts` в
[SGLang-Omni](https://github.com/sgl-project/sglang-omni), она же указана в model card.
Локальный замер на M1 идёт через совершенно другую реализацию — MLX-Audio, независимый
порт Higgs v3 на MLX. То, что синтез работает на Apple GPU, ничего не говорит о CUDA:
это разный код.

**Порог по железу нигде не задокументирован.** SGLang-Omni не объявляет минимальную
compute capability, но пинит колёса `flash-attn-4` и `flashinfer`, рассчитанные на
свежие архитектуры, а `higgs_tts/sampler.py` вызывает renorm-ядра flashinfer. Отсюда
ожидание, что на T4 (compute 7.5) стек не заведётся — но это **ожидание, а не
проверенный факт**: у SGLang есть бэкенды `triton` и `torch_native`.

Поэтому блокнот **не отказывается запускаться заранее**. Он предупреждает, пытается и
записывает то, что действительно произошло, вместе с логом сервера. Так появляется
фактическое доказательство вместо догадки. Если жечь квоту GPU на вероятный отказ не
хочется, выставьте `TTS_MIN_CAPABILITY = "8.9"` — тогда этап честно пропустится.

Для наибольших шансов на успех выбирайте **Runtime → Change runtime type → L4 / A100**.

## 1. Настройки запуска

In [ ]:
# ── Флаги запуска ───────────────────────────────────────────
USE_DRIVE        = True    # хранить входы/выходы на Google Drive
RUN_STT          = True
RUN_TTS          = True
INSTALL_TTS_STACK = True   # ставить SGLang-Omni (10–20 минут)
AUTO_DISCONNECT  = False   # отключать ВМ по завершении (по умолчанию выключено,
                           # чтобы вывод Run all оставался читаемым)

REPO_URL = "https://github.com/vedmalex/higgs-local-test.git"
REPO_REF = "main"          # ветка/тег/SHA репозитория с раннерами

STT_DTYPE = "float16"      # float16 | bfloat16 | float32
TTS_MAX_NEW_TOKENS = 2048
TTS_MEM_FRACTION_STATIC = None   # напр. 0.80, чтобы ограничить статический пул VRAM
TTS_MIN_CAPABILITY = None        # напр. "8.9" — пропустить TTS на более старом GPU,
                                 # не тратя время на вероятный отказ
TTS_SERVER_ARGS = []             # доп. аргументы sgl-omni, напр.
                                 # ["attention_backend=triton"]

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path

def run(command, **kwargs):
    """Запуск дочернего процесса с потоковым выводом. Возвращает returncode."""
    printable = command if isinstance(command, str) else " ".join(shlex.quote(c) for c in command)
    print(f"$ {printable}")
    process = subprocess.Popen(command, shell=isinstance(command, str),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, **kwargs)
    for line in process.stdout:
        print(line, end="")
    return process.wait()

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/higgs-benchmark")
else:
    WORKSPACE = Path("/content/higgs-benchmark")

SAMPLES_DIR = WORKSPACE / "samples"
OUTPUT_DIR  = WORKSPACE / "output"
METRICS_DIR = WORKSPACE / "metrics"
for directory in (SAMPLES_DIR, OUTPUT_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Веса моделей живут на локальном диске ВМ, а не на Drive: 15 ГБ через FUSE
# загружаются несравнимо медленнее и съедают квоту Диска.
HF_HOME = Path("/content/hf-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

REPO = Path("/content/higgs-local-test")
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)])
REPO_SHA = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip()

print(f"\nРабочая область: {WORKSPACE}")
print(f"Кэш весов:       {HF_HOME}")
print(f"Раннеры:         {REPO} @ {REPO_SHA[:12]}")

for directory, label in ((SAMPLES_DIR, "Входные сэмплы"), (OUTPUT_DIR, "Результаты")):
    files = [f for f in sorted(directory.iterdir()) if f.is_file() and not f.name.startswith(".")]
    print(f"\n{label} ({directory}): {len(files)} файлов")
    for f in files:
        print(f"  - {f.name} ({f.stat().st_size / (1024 ** 2):.2f} MB)")

## 2. GPU и вход

Проверка выделенного GPU и наличия входных данных. Отсутствующий вход даёт `SKIPPED` —
синтетический сигнал вместо записи речи не подставляется, потому что распознавание
синуса дало бы бессмысленные RTF и WER.

In [ ]:
run(["nvidia-smi"])

import torch  # предустановлен в Colab; переустановка сломала бы связку CUDA/torchvision

GPU = None
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU = {"name": props.name, "capability": (props.major, props.minor),
           "total_memory_gb": props.total_memory / (1024 ** 3),
           "bf16": torch.cuda.is_bf16_supported()}
    print(f"\nGPU: {GPU['name']} | compute {props.major}.{props.minor} | "
          f"VRAM {GPU['total_memory_gb']:.1f} GB | bf16: {GPU['bf16']}")
    print(f"torch: {torch.__version__}")
else:
    print("\n⚠️  CUDA-устройство недоступно: Runtime → Change runtime type → GPU")

# ── Вход для STT ─────────────────────────────────────────────
STT_INPUT = SAMPLES_DIR / "stt_ru.wav"
STT_REFERENCE = SAMPLES_DIR / "stt_ru.txt"   # необязательный референс для WER
if not STT_INPUT.exists():
    print(f"\nℹ️  STT будет SKIPPED: положите русскую запись в {STT_INPUT}")
else:
    print(f"\nSTT вход: {STT_INPUT}")
    if STT_REFERENCE.exists():
        print(f"WER считается против {STT_REFERENCE}")
    else:
        print("WER считается против встроенного фикстур-текста репозитория — "
              "он валиден только для samples/stt_ru.wav из репозитория. "
              f"Для своей записи положите её точную расшифровку в {STT_REFERENCE}.")

# ── Вход для TTS ─────────────────────────────────────────────
TTS_TEXT = SAMPLES_DIR / "tts_ru.txt"
if not TTS_TEXT.exists():
    TTS_TEXT = REPO / "samples/tts_ru.txt"
REF_WAV, REF_TXT = SAMPLES_DIR / "reference.wav", SAMPLES_DIR / "reference.txt"
print(f"\nTTS текст: {TTS_TEXT}")
print("Клонирование голоса: "
      + ("эталон найден" if REF_WAV.exists() and REF_TXT.exists()
         else f"SKIPPED — нужны {REF_WAV.name} и {REF_TXT.name} в {SAMPLES_DIR}"))

## 3. STT: `bosonai/higgs-audio-v3-stt`

Окружение создаётся с `--system-site-packages`, поэтому предустановленный в Colab
`torch` переиспользуется, а в venv доустанавливается только несовместимый с системным
`transformers==4.51.0` и мелкие зависимости. Ревизия чекпоинта закреплена в
`src/stt_helper.py`.

In [ ]:
STT_VENV = Path("/content/venv-stt")
STT_PY = STT_VENV / "bin/python"

def python_paths(interpreter) -> list:
    """Пути импорта, которые реально видит указанный интерпретатор."""
    probe = "import json, sys; print(json.dumps(sys.path))"
    result = subprocess.run([str(interpreter), "-c", probe], capture_output=True, text=True)
    result.check_returncode()
    return json.loads(result.stdout)

def link_system_packages(interpreter) -> None:
    """Открыть venv доступ к системным пакетам Colab.

    Colab держит torch в `dist-packages`, и `venv --system-site-packages` их не
    подхватывает: Debian-раскладка отличается от той, которую ожидает `site.py`
    внутри venv. Поэтому пути родительского интерпретатора добавляются через
    `.pth`-файл. Он дописывается в КОНЕЦ `sys.path`, поэтому пакеты самого venv
    (в частности transformers 4.51.0) сохраняют приоритет над системными.
    """
    inherited = [p for p in python_paths(sys.executable)
                 if p and Path(p).is_dir() and not str(p).startswith(str(STT_VENV))]
    site_dir = subprocess.run(
        [str(interpreter), "-c", "import site; print(site.getsitepackages()[0])"],
        capture_output=True, text=True, check=True).stdout.strip()
    pth = Path(site_dir) / "_colab_system_packages.pth"
    pth.write_text("\n".join(inherited) + "\n", encoding="utf-8")
    print(f"Системные пути Colab подключены через {pth} ({len(inherited)} каталогов)")

if RUN_STT and not STT_PY.exists():
    run([sys.executable, "-m", "venv", "--system-site-packages", str(STT_VENV)])
    link_system_packages(STT_PY)
    run([str(STT_PY), "-m", "pip", "install", "-q", "--upgrade", "pip"])
    # transformers 4.51.0 — версия, объявленная в config.json чекпоинта STT.
    # Ставится в venv, поэтому системный transformers остаётся нетронутым.
    run([str(STT_PY), "-m", "pip", "install", "-q",
         "transformers==4.51.0", "tokenizers<0.22", "accelerate>=0.26.0",
         "huggingface_hub<1.0", "soundfile", "librosa", "jiwer", "sentencepiece"])

elif RUN_STT:
    # venv остался от прошлого запуска — дошиваем .pth, если его там нет.
    link_system_packages(STT_PY)

if RUN_STT:
    probe = ("import torch, transformers, huggingface_hub as hub; "
             "print('torch', torch.__version__, '(', torch.__file__, ')'); "
             "print('transformers', transformers.__version__, '| hub', hub.__version__, "
             "'| cuda', torch.cuda.is_available())")
    if run([str(STT_PY), "-c", probe]) != 0:
        raise RuntimeError(
            "Окружение STT не видит torch. Проверьте, что ячейка выполняется в Colab "
            "с предустановленным torch, и что link_system_packages() записал .pth. "
            f"Пути родительского интерпретатора: {python_paths(sys.executable)}")

In [ ]:
STT_METRICS = METRICS_DIR / "stt_cuda.json"
stt_status = "NOT RUN"

if not RUN_STT:
    stt_status = "NOT RUN (RUN_STT=False)"
elif GPU is None:
    stt_status = "SKIPPED (нет CUDA-устройства)"
elif not STT_INPUT.exists():
    stt_status = f"SKIPPED (нет входа {STT_INPUT})"
else:
    # ffmpeg предустановлен в Colab; чекпоинт ожидает моно 16 кГц.
    normalized = Path("/content/stt_ru_16k.wav")
    if run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(STT_INPUT),
            "-ac", "1", "-ar", "16000", str(normalized)]) != 0:
        raise RuntimeError(f"ffmpeg не смог нормализовать {STT_INPUT} в моно 16 кГц")

    command = [str(STT_PY), "src/stt_test.py",
               "--device", "cuda", "--dtype", STT_DTYPE,
               "--audio", str(normalized),
               "--output", str(OUTPUT_DIR / "stt_ru_colab.txt"),
               "--metrics", str(STT_METRICS)]
    if STT_REFERENCE.exists():
        command += ["--reference", str(STT_REFERENCE)]

    # Отдельный процесс: по его завершении вся VRAM и RAM модели возвращаются системе.
    returncode = run(command, cwd=str(REPO))
    stt_status = "PASSED" if returncode == 0 else f"FAILED (exit {returncode})"

print(f"\nSTT: {stt_status}")
if torch.cuda.is_available():
    # Ядро блокнота не аллоцировало модель, поэтому здесь ожидается около нуля.
    print(f"VRAM, удерживаемая ядром блокнота: "
          f"{torch.cuda.memory_allocated() / (1024 ** 3):.3f} GB")
run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

In [ ]:
stt_metrics = json.loads(STT_METRICS.read_text(encoding="utf-8")) if STT_METRICS.exists() else None

if stt_metrics and stt_metrics.get("status") == "PASSED":
    print("📝 Транскрипция:\n" + stt_metrics["transcript"] + "\n")
    print(f"Устройство:      {stt_metrics['cuda_device']} ({stt_metrics['dtype']})")
    print(f"Ревизия модели:  {stt_metrics['revision']}")
    print(f"Загрузка:        {stt_metrics['model_load_seconds']:.2f} с")
    print(f"Длит. аудио:     {stt_metrics['audio_duration_seconds']:.2f} с")
    print(f"Обработка:       {stt_metrics['processing_seconds']:.2f} с")
    print(f"RTF:             {stt_metrics['rtf']:.3f}")
    print(f"WER:             {stt_metrics['wer']:.4f} (референс: {stt_metrics['wer_reference']})")
    print(f"Пик VRAM:        {stt_metrics['peak_vram_bytes'] / (1024 ** 3):.2f} GB")
    print(f"Пик RSS:         {stt_metrics['peak_host_rss_bytes'] / (1024 ** 3):.2f} GB")
elif stt_metrics:
    print(f"STT {stt_metrics.get('status')}: {stt_metrics.get('exception')}")
    print(stt_metrics.get("traceback", ""))
else:
    print(f"STT метрик нет: {stt_status}")

## 4. TTS: `bosonai/higgs-tts-3-4b` через SGLang-Omni

`src/tts_cuda.py` сам проверяет пригодность GPU и наличие `sgl-omni`, поднимает сервер
отдельной группой процессов, синтезирует три варианта и гасит сервер — вместе с ним
уходит вся занятая им VRAM. Если требование не выполнено, раннер пишет `SKIPPED` с
причиной и не выдаёт никаких чисел.

Установка стека занимает 10–20 минут. На GPU ниже Ada выводится предупреждение, но
запуск всё равно выполняется: реальный отказ с логом полезнее моего предположения.

In [ ]:
TTS_VENV = Path("/content/venv-tts")
TTS_PY = TTS_VENV / "bin/python"
TTS_BIN = TTS_VENV / "bin"

tts_gate = None
if not RUN_TTS:
    tts_gate = "RUN_TTS=False"
elif GPU is None:
    tts_gate = "нет CUDA-устройства"
elif TTS_MIN_CAPABILITY and GPU["capability"] < tuple(
        int(part) for part in f"{TTS_MIN_CAPABILITY}.0".split(".")[:2]):
    tts_gate = (f"{GPU['name']} имеет compute {GPU['capability'][0]}.{GPU['capability'][1]}, "
                f"ниже заданного TTS_MIN_CAPABILITY={TTS_MIN_CAPABILITY}")
elif GPU["capability"] < (8, 9):
    # Предупреждение, а не отказ: порог документирован не производителем стека, а
    # выведен из его пинов, поэтому проверяем на практике и фиксируем результат.
    print(f"⚠️  {GPU['name']} имеет compute {GPU['capability'][0]}.{GPU['capability'][1]}. "
          "Стек SGLang-Omni на таком GPU не проверен и может упасть при установке или "
          "старте сервера. Запуск всё равно состоится, отказ будет зафиксирован с логом. "
          "Может помочь TTS_SERVER_ARGS = ['attention_backend=triton'].")

SGL_OMNI = TTS_BIN / "sgl-omni"

if tts_gate:
    print(f"TTS стек не устанавливается: {tts_gate}")
elif INSTALL_TTS_STACK and not SGL_OMNI.exists():
    # Без --system-site-packages: SGLang-Omni пинит собственные torch/transformers,
    # которые нельзя смешивать ни с системными, ни с STT-окружением.
    steps = [
        [sys.executable, "-m", "venv", str(TTS_VENV)],
        [str(TTS_PY), "-m", "pip", "install", "-q", "--upgrade", "pip", "wheel"],
        # 0.1.3 — версия, в которой проверено наличие models/higgs_tts.
        [str(TTS_PY), "-m", "pip", "install", "sglang-omni==0.1.3"],
    ]
    for step in steps:
        # Код возврата обязателен: молча проигнорированная ошибка pip раньше
        # приводила к тому, что этап сообщал «torch is not installed» вместо
        # настоящей причины — неустановленного стека.
        returncode = run(step)
        if returncode != 0:
            tts_gate = (f"установка стека SGLang-Omni не удалась: "
                        f"`{' '.join(step[-2:])}` завершилась с кодом {returncode}. "
                        "Причина — в выводе pip выше.")
            print(f"\n❌ {tts_gate}")
            break
    else:
        print("\n✅ Стек SGLang-Omni установлен")
elif SGL_OMNI.exists():
    print("TTS окружение уже создано")
else:
    tts_gate = ("окружение SGLang-Omni отсутствует, а INSTALL_TTS_STACK=False"
                if not INSTALL_TTS_STACK else
                "в окружении нет исполняемого файла sgl-omni: установка не завершилась")
    print(f"TTS: {tts_gate}")

In [ ]:
TTS_METRICS = METRICS_DIR / "tts_cuda.json"
tts_status = "NOT RUN"

if tts_gate:
    tts_status = f"SKIPPED ({tts_gate})"
    TTS_METRICS.write_text(json.dumps(
        {"test": "tts_cuda", "status": "SKIPPED", "reason": tts_gate, "results": []},
        ensure_ascii=False, indent=2), encoding="utf-8")
elif not SGL_OMNI.exists():
    tts_status = "SKIPPED (в окружении нет sgl-omni)"
    TTS_METRICS.write_text(json.dumps(
        {"test": "tts_cuda", "status": "SKIPPED",
         "reason": "sgl-omni is absent from the TTS environment", "results": []},
        ensure_ascii=False, indent=2), encoding="utf-8")
else:
    command = [str(TTS_PY), "src/tts_cuda.py",
               "--output-dir", str(OUTPUT_DIR),
               "--metrics", str(TTS_METRICS),
               "--text-file", str(TTS_TEXT),
               "--max-new-tokens", str(TTS_MAX_NEW_TOKENS)]
    if REF_WAV.exists() and REF_TXT.exists():
        command += ["--ref-audio", str(REF_WAV), "--ref-text", str(REF_TXT)]
    if TTS_MEM_FRACTION_STATIC is not None:
        command += ["--mem-fraction-static", str(TTS_MEM_FRACTION_STATIC)]
    if TTS_MIN_CAPABILITY:
        command += ["--min-capability", str(TTS_MIN_CAPABILITY)]
    for server_arg in TTS_SERVER_ARGS:
        command += ["--server-arg", server_arg]

    # PATH нужен, чтобы раннер нашёл `sgl-omni` своего venv.
    environment = {**os.environ, "PATH": f"{TTS_BIN}:{os.environ['PATH']}"}
    returncode = run(command, cwd=str(REPO), env=environment)
    tts_status = "PASSED" if returncode == 0 else f"FAILED (exit {returncode})"

print(f"\nTTS: {tts_status}")
run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

In [ ]:
from IPython.display import Audio, display

tts_metrics = json.loads(TTS_METRICS.read_text(encoding="utf-8")) if TTS_METRICS.exists() else None

if not tts_metrics:
    print(f"TTS метрик нет: {tts_status}")
elif tts_metrics.get("status") == "SKIPPED":
    print(f"TTS SKIPPED: {tts_metrics['reason']}")
else:
    if tts_metrics.get("status") == "FAILED":
        print(f"TTS FAILED: {tts_metrics.get('exception')}")
        server_log = Path(tts_metrics.get("server_log") or (OUTPUT_DIR / "sgl_omni_server.log"))
        if server_log.exists():
            print(f"\n--- последние строки {server_log.name} ---")
            print("\n".join(server_log.read_text(errors="replace").splitlines()[-40:]))

    for result in tts_metrics.get("results", []):
        if result["status"] != "PASSED":
            print(f"{result['name']}: {result['status']} — "
                  f"{result.get('reason') or result.get('exception')}")
            continue
        print(f"{result['name']}: {result['audio_duration_seconds']:.2f} с аудио за "
              f"{result['processing_seconds']:.2f} с → RTF {result['rtf']:.3f}")
        display(Audio(filename=result["output"]))

## 5. Сводка

Таблица собирается только из записанных метрик. Пустых мест нет: там, где этап не
выполнялся, стоит его фактический статус, а не подставленное значение.

In [ ]:
import pandas as pd

# Опорные значения локального замера на Apple Silicon M1 (16 ГБ, macOS 14.6.1).
M1_RTF = {"stt": 1.40, "tts_basic": 7.02, "tts_controls": 12.61, "tts_clone": 822.09}

rows = []

if stt_metrics and stt_metrics.get("status") == "PASSED":
    rows.append({"Этап": "STT", "Статус": "PASSED",
                 "RTF (Colab)": f"{stt_metrics['rtf']:.3f}",
                 "RTF (M1)": f"{M1_RTF['stt']:.2f}",
                 "Пик VRAM, GB": f"{stt_metrics['peak_vram_bytes'] / (1024 ** 3):.2f}",
                 "Артефакт": stt_metrics["output"]})
else:
    rows.append({"Этап": "STT", "Статус": stt_status, "RTF (Colab)": "—",
                 "RTF (M1)": f"{M1_RTF['stt']:.2f}", "Пик VRAM, GB": "—", "Артефакт": "—"})

tts_results = {r["name"]: r for r in (tts_metrics or {}).get("results", [])}
for name in ("tts_basic", "tts_controls", "tts_clone"):
    result = tts_results.get(name)
    if result and result["status"] == "PASSED":
        rows.append({"Этап": name, "Статус": "PASSED",
                     "RTF (Colab)": f"{result['rtf']:.3f}",
                     "RTF (M1)": f"{M1_RTF[name]:.2f}",
                     # Веса TTS живут в процессе sgl-omni, поэтому пик замеряется
                     # device-wide через nvidia-smi, а не через torch этого процесса.
                     "Пик VRAM, GB": (f"{tts_metrics['peak_device_vram_bytes'] / (1024 ** 3):.2f}*"
                                      if tts_metrics.get("peak_device_vram_bytes") else "—"),
                     "Артефакт": result["output"]})
    else:
        status = (result or {}).get("status") or (tts_metrics or {}).get("status") or tts_status
        reason = (result or {}).get("reason") or (tts_metrics or {}).get("reason") or ""
        rows.append({"Этап": name, "Статус": f"{status}{': ' + reason if reason else ''}",
                     "RTF (Colab)": "—", "RTF (M1)": f"{M1_RTF[name]:.2f}",
                     "Пик VRAM, GB": "—", "Артефакт": "—"})

summary = pd.DataFrame(rows)
display(summary)
print("STT: пик VRAM — аллокации процесса STT (torch). "
      "TTS (*): пик по устройству целиком (nvidia-smi), пока жил сервер sgl-omni.")

report = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S%z"),
    "platform": "Google Colab",
    "repo_revision": REPO_SHA,
    "gpu": GPU,
    "notebook_torch": torch.__version__,
    "stt": stt_metrics or {"status": stt_status},
    "tts": tts_metrics or {"status": tts_status},
}
report_path = METRICS_DIR / "benchmark_colab_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"\n✅ Полный отчёт: {report_path}")

In [ ]:
# ── Завершение ──────────────────────────────────────────────
# Модели жили в дочерних процессах, которые уже завершились, поэтому выгружать
# из ядра нечего. Остаётся только синхронизировать Drive.
if USE_DRIVE:
    from google.colab import drive
    drive.flush_and_unmount()
    print("💾 Файлы синхронизированы с Google Drive: MyDrive/higgs-benchmark/")

if AUTO_DISCONNECT:
    from google.colab import runtime
    print("🛑 Отключение ВМ для экономии квоты...")
    runtime.unassign()
else:
    print("ℹ️  ВМ оставлена включённой. Для авто-отключения выставьте AUTO_DISCONNECT = True.")